In [2]:
import os
import pickle
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.models import load_model
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# === Load test data ===
x_test = pd.read_csv("ds_34_x_test.csv").iloc[:, 1:].values
y_test = pd.read_csv("ds_34_y_test.csv").iloc[:, 1].values.ravel()

y_test = y_test - 1  # Adjust if labels start from 1

# === Path to saved models folder ===
models_path = "all_models/"
model_files = os.listdir(models_path)

results = []

print("\nEvaluating saved models...\n")

for file in model_files:
    file_path = os.path.join(models_path, file)

    # === Load model depending on file format ===
    try:
        if file.endswith((".pkl", ".sav")):
            with open(file_path, "rb") as f:
                model = pickle.load(f)
        elif file.endswith(".joblib"):
            model = joblib.load(file_path)
        elif file.endswith((".h5", ".keras")):
            model = load_model(file_path)
        else:
            print(f"Skipping unknown format: {file}")
            continue
    except Exception as e:
        print(f"Error loading {file}: {e}")
        continue

    # === Make predictions ===
    try:
        if file.endswith((".h5", ".keras")):  # Handle Keras/TensorFlow models
            y_pred = model.predict(x_test)

            # Flatten predictions for binary classification
            if y_pred.ndim > 1 and y_pred.shape[1] == 1:
                y_pred = (y_pred.ravel() > 0.5).astype("int32")
            # Multi-class classification
            elif y_pred.ndim > 1:
                y_pred = y_pred.argmax(axis=1)
            else:
                y_pred = (y_pred > 0.5).astype("int32")
        else:
            y_pred = model.predict(x_test)

        # === Debugging: Check shapes ===
        print(f"\nModel: {file}")
        print("x_test shape:", x_test.shape)
        print("y_test shape:", y_test.shape)
        print("y_pred shape:", y_pred.shape)

        # === Shape alignment check ===
        if len(y_pred) != len(y_test):
            print(f"Skipping {file}: shape mismatch ({len(y_test)} vs {len(y_pred)})")
            continue

        # === Compute metrics ===
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
        rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
        f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

        results.append([file, acc, prec, rec, f1])

    except Exception as e:
        print(f"Error processing {file}: {e}")
        continue

# === Create DataFrame of results ===
if not results:
    print("\nNo models were successfully evaluated. Check your file paths or data shapes.")
else:
    results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "Precision", "Recall", "F1 Score"])
    results_df = results_df.sort_values(by="Accuracy", ascending=False)

    print("\n==============================")
    print("   MODEL PERFORMANCE SUMMARY   ")
    print("==============================")
    print(results_df.to_string(index=False))

    # Save results to CSV
    results_df.to_csv("all_model_comparison_summary.csv", index=False)
    print("\nResults saved to 'all_model_comparison_summary.csv'")

    # === Visualization ===
    sns.set(style="whitegrid", font_scale=1.1)
    plt.figure(figsize=(10, 6))

    metrics = ["Accuracy", "Precision", "Recall", "F1 Score"]


Evaluating saved models...

253/253 [==============================] - 1s 2ms/step

Model: ann_2layer_model_s.keras
x_test shape: (8066, 27)
y_test shape: (8066,)
y_pred shape: (8066,)
253/253 [==============================] - 1s 2ms/step

Model: ann_3layer_model_s.keras
x_test shape: (8066, 27)
y_test shape: (8066,)
y_pred shape: (8066,)
253/253 [==============================] - 1s 4ms/step

Model: cnn_model_s.h5
x_test shape: (8066, 27)
y_test shape: (8066,)
y_pred shape: (8066,)

Model: gradient_boost_model_s.pkl
x_test shape: (8066, 27)
y_test shape: (8066,)
y_pred shape: (8066,)

Model: knn_model_s.pkl
x_test shape: (8066, 27)
y_test shape: (8066,)
y_pred shape: (8066,)

Model: random_forest_model_s.pkl
x_test shape: (8066, 27)
y_test shape: (8066,)
y_pred shape: (8066,)

Model: svm_rbf_model_s.pkl
x_test shape: (8066, 27)
y_test shape: (8066,)
y_pred shape: (8066,)

Model: xgboost_model_s.pkl
x_test shape: (8066, 27)
y_test shape: (8066,)
y_pred shape: (8066,)

   MODEL PERFOR

<Figure size 1000x600 with 0 Axes>